# L2 vs OT (Sinkhorn) Data-Loss Comparison

## Description

- **Purpose**: Compare **L2 (MSE)** and **OT (Sinkhorn)** as data misfit in 1D MT inversion under the same true model and the same synthetic dataset.
- **True model**: The **geomloss-style** 6-layer model is used for both runs: thickness `dz = [600, 600, 600, 600, 600]` m, conductivity `sig = [0.1, 0.08, 0.3, 0.05, 0.01, 0.005]` S/m. L2 and OT share this model and the same noisy data (same `seed`).
- **Stopping criterion**: Automatic stop when **χ² RMS < 1.05** (`target_rms=1.05`, `enable_auto_stop=True`).
- **Parameters**: L2 and OT settings are taken from the *MT1D L2 (test)* and *MT1D geomloss* notebooks respectively.
- **Plotting**: Both inversions use MTinv plotting functions—`plot_synthetic_data`, `plot_data_fit`, `plot_loss_history`, `plot_model_comparison`, `plot_chi2_history`, `plot_gradient_history`, `plot_sensitivity`—for comparing fit and model recovery.

In [ ]:
import sys
sys.path.insert(0, '..')
import torch
import numpy as np
import matplotlib.pyplot as plt

from src.mt1d_inv import MT1DInverter
from src.mt1d_inv.model import MT1D

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# True model: geomloss-style 6-layer
dz_true = [600, 600, 600, 600, 600]
sig_true = [0.1, 0.08, 0.3, 0.05, 0.01, 0.005]
my_model = MT1D(dz_true, sig_true)
true_dz = my_model.dz
true_sig = my_model.sig
print("True model: dz =", dz_true, ", sig =", sig_true)

## 1. L₂ (MSE) inversion

In [ ]:
# L2 inverter (parameters from L2 test)
inv_l2 = MT1DInverter(device=device, use_sinkhorn=False)
inv_l2.generate_synthetic_data(
    true_dz=true_dz,
    true_sig=true_sig,
    freq_range=(1, -4),
    n_freq=60,
    noise_level=0.05,
    seed=42
)
inv_l2.initialize_model(
    n_layers=20,
    total_depth=3000,
    initial_sig=0.1,
    thickness_mode="equal"
)
inv_l2.setup_constraints(use_occam_constraint=True, constraint_type="roughness")
inv_l2.setup_optimizer(lr=0.01, reg_weight_sig=0.1, p=2)

inv_l2.run_inversion(
    num_epochs=1000,
    print_interval=50,
    seed=42,
    track_chi2=True,
    enable_auto_stop=True,
    target_rms=1.05
)

### L2 results: plotting functions

MTinv plotting API: synthetic data, data fit, loss history, model comparison, χ² history, gradient history, sensitivity.

In [ ]:
inv_l2.plot_synthetic_data()
inv_l2.plot_data_fit()
inv_l2.plot_loss_history(target_misfit=1.0)
inv_l2.plot_model_comparison()
inv_l2.plot_chi2_history()
inv_l2.plot_gradient_history()
inv_l2.plot_sensitivity()

## 2. OT (Sinkhorn) inversion


In [ ]:
# OT inverter (parameters from geomloss notebook)
inv_ot = MT1DInverter(device=device, use_sinkhorn=True, sinkhorn_dim=3)
inv_ot.generate_synthetic_data(
    true_dz=true_dz,
    true_sig=true_sig,
    freq_range=(1, -4),
    n_freq=60,
    noise_level=0.05,
    seed=42
)
inv_ot.initialize_model(
    n_layers=20,
    total_depth=3000,
    initial_sig=0.01,
    thickness_mode="equal"
)
inv_ot.setup_constraints(use_occam_constraint=True, constraint_type="roughness")
inv_ot.setup_optimizer(
    lr=0.05,
    reg_weight_sig=0.0001,
    optimizer_type="AdamW",
    weight_decay=0.01,
    betas=(0.9, 0.999)
)
inv_ot.set_reference_model([0.01] * 10, weight=0.01)

inv_ot.run_inversion(
    num_epochs=3000,
    print_interval=50,
    seed=42,
    track_chi2=True,
    enable_auto_stop=True,
    target_rms=1.05
)

### OT results: plotting functions

In [ ]:
inv_ot.plot_synthetic_data()
inv_ot.plot_data_fit()
inv_ot.plot_loss_history(target_misfit=1.0)
inv_ot.plot_model_comparison()
inv_ot.plot_chi2_history()
inv_ot.plot_gradient_history()
inv_ot.plot_sensitivity()